# 09. K-Means: Descubriendo Grupos en Datos

**Nivel:** 🟢 Principiante  
**Tiempo estimado:** 60 minutos  
**Prerequisitos:** Conceptos básicos de distancias y vectores

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Entender el concepto de clustering y aprendizaje no supervisado
- Implementar K-Means desde cero usando solo NumPy
- Comprender el algoritmo iterativo: asignar-actualizar-repetir
- Usar el método del codo para elegir K óptimo
- Aplicar K-Means++ para mejor inicialización
- Identificar limitaciones y casos de uso apropiados

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, load_iris, load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Importar utilidades
import sys
sys.path.append('../../shared/utils')
from visualization import plot_clusters
from datasets import load_dataset

np.random.seed(42)
print("✅ Librerías importadas")

---
## 📌 1. Motivación: Aprendizaje Sin Etiquetas

### El Gran Cambio: Supervisado → No Supervisado

**Hasta ahora (Supervisado):**
```python
Datos: X = features, y = labels
Objetivo: Aprender f: X → y
Ejemplo: "Esta imagen es un gato" ✅
```

**Ahora (No Supervisado):**
```python
Datos: X = features (¡sin labels!)
Objetivo: Descubrir estructura oculta
Ejemplo: "Estas imágenes se parecen entre sí" 🤔
```

### ¿Qué es Clustering?

**Agrupar datos similares juntos.**

```
Antes:                    Después:
  🔴 🔵 🟢                  [🔴 🔴 🔴]
🔵 🔴 🟢 🔴                [🔵 🔵 🔵]
  🟢 🔵 🔴                  [🟢 🟢 🟢]
(datos mezclados)         (3 clusters)
```

### K-Means: La Idea Central

**1. Elige K (número de clusters)**
```
K=3 → Buscaré 3 grupos
```

**2. Inicializa K centroides aleatorios**
```
      ⭐                ⭐ = Centroide
  🔴🔵🟢 ⭐ 🔴🔵          🔴🔵🟢 = Puntos
    ⭐  🟢
```

**3. Repite hasta convergencia:**
```
a) Asignar: Cada punto al centroide más cercano
   🔴 → ⭐₁ (más cercano)
   🔵 → ⭐₂
   🟢 → ⭐₃

b) Actualizar: Mover centroides al centro de sus puntos
   ⭐₁ = promedio(todos los 🔴 asignados)
```

### Aplicaciones Reales

- 🛒 **Segmentación de clientes**: Agrupar por comportamiento de compra
- 📰 **Agrupación de noticias**: Temas similares juntos
- 🧬 **Análisis genético**: Identificar subpoblaciones
- 🖼️ **Compresión de imágenes**: Reducir paleta de colores
- 🗺️ **Análisis geoespacial**: Identificar zonas con características similares
- 🎵 **Recomendación de música**: Usuarios con gustos similares

### Problema Real: Segmentación de Clientes

Tienes datos de 10,000 clientes:
- Edad
- Ingresos anuales
- Gasto promedio

**Objetivo:** Agruparlos en segmentos para:
- Campañas de marketing personalizadas
- Identificar clientes VIP
- Entender patrones de comportamiento

**K-Means puede revelar:**
- Cluster 1: Jóvenes, bajo ingreso, bajo gasto
- Cluster 2: Edad media, alto ingreso, alto gasto (VIP!)
- Cluster 3: Mayores, ingreso medio, gasto selectivo

### La Pregunta Guía

> **¿Cómo podemos encontrar grupos naturales en datos sin saber de antemano qué buscamos?**

---
## 📊 2. Intuición Visual

In [ ]:
# Generar datos con clusters naturales
X, y_true = make_blobs(n_samples=300, centers=4, n_features=2,
                      cluster_std=0.60, random_state=42)

# Visualizar datos sin labels
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=X[:, 0],
    y=X[:, 1],
    mode='markers',
    marker=dict(size=8, color='gray', opacity=0.6),
    name='Datos (sin etiquetas)'
))

fig.update_layout(
    title="Datos Sin Etiquetas: ¿Puedes Ver los Grupos?",
    xaxis_title="Feature 1",
    yaxis_title="Feature 2",
    template="plotly_white",
    font=dict(size=12),
    width=700,
    height=500
)

fig.show()

print("\n👁️ ¿Puedes identificar grupos visualmente?")
print("   K-Means hará esto automáticamente")

In [ ]:
# Aplicar K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)
centroids = kmeans.cluster_centers_

# Visualizar resultado
fig = go.Figure()

# Puntos coloreados por cluster
colors = ['red', 'blue', 'green', 'orange']
for i in range(4):
    mask = clusters == i
    fig.add_trace(go.Scatter(
        x=X[mask, 0],
        y=X[mask, 1],
        mode='markers',
        marker=dict(size=8, color=colors[i], opacity=0.6),
        name=f'Cluster {i+1}'
    ))

# Centroides
fig.add_trace(go.Scatter(
    x=centroids[:, 0],
    y=centroids[:, 1],
    mode='markers',
    marker=dict(size=20, color='black', symbol='x', line=dict(width=2)),
    name='Centroides'
))

fig.update_layout(
    title="K-Means: Datos Agrupados en 4 Clusters",
    xaxis_title="Feature 1",
    yaxis_title="Feature 2",
    template="plotly_white",
    font=dict(size=12),
    width=700,
    height=500
)

fig.show()

print("\n💡 Observa:")
print("   • Cada color es un cluster")
print("   • Las X negras son los centroides (centros de los clusters)")
print("   • Cada punto está asignado al centroide más cercano")

In [ ]:
# Animación del proceso iterativo de K-Means
# (Versión simplificada con 3 iteraciones)

def visualize_kmeans_iteration(X, n_clusters=4, n_iterations=5):
    """Visualiza las iteraciones de K-Means"""
    
    # Inicialización aleatoria
    np.random.seed(42)
    indices = np.random.choice(len(X), n_clusters, replace=False)
    centroids = X[indices].copy()
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()
    
    colors = ['red', 'blue', 'green', 'orange']
    
    for iteration in range(min(n_iterations, 6)):
        # Asignar clusters
        distances = cdist(X, centroids)
        clusters = np.argmin(distances, axis=1)
        
        # Plot
        for i in range(n_clusters):
            mask = clusters == i
            axes[iteration].scatter(X[mask, 0], X[mask, 1], 
                                   c=colors[i], alpha=0.6, s=50)
        
        axes[iteration].scatter(centroids[:, 0], centroids[:, 1],
                               c='black', marker='x', s=200, linewidths=3)
        axes[iteration].set_title(f'Iteración {iteration + 1}', fontsize=14)
        axes[iteration].set_xlabel('Feature 1')
        axes[iteration].set_ylabel('Feature 2')
        
        # Actualizar centroides
        new_centroids = np.array([X[clusters == i].mean(axis=0) 
                                 for i in range(n_clusters)])
        centroids = new_centroids
    
    plt.tight_layout()
    plt.show()

visualize_kmeans_iteration(X, n_clusters=4, n_iterations=6)

print("\n💡 Observa cómo:")
print("   1. Los centroides se mueven en cada iteración")
print("   2. Las asignaciones de clusters cambian")
print("   3. Eventualmente converge (centroides ya no se mueven)")

---
## 🧮 3. Fundamentos Matemáticos

### 📖 Notación

| Símbolo | Significado |
|---------|-------------|
| $K$ | Número de clusters |
| $\mathbf{x}_i$ | Punto de datos $i$ |
| $\mu_k$ | Centroide del cluster $k$ |
| $C_k$ | Conjunto de puntos en cluster $k$ |
| $d(\cdot, \cdot)$ | Función de distancia (típicamente Euclidiana) |

---

## 3.1 Función Objetivo

K-Means minimiza la **inercia** (suma de distancias al cuadrado dentro de clusters):

$$
J = \sum_{k=1}^{K} \sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \mu_k\|^2 \tag{1}
$$

**Interpretación:** Queremos que los puntos estén lo más cerca posible de sus centroides.

### Distancia Euclidiana

$$
d(\mathbf{x}_i, \mu_k) = \sqrt{\sum_{j=1}^{d} (x_{ij} - \mu_{kj})^2} \tag{2}
$$

Donde $d$ es el número de dimensiones (features).

---

## 3.2 Algoritmo K-Means (Lloyd's Algorithm)

**Input:** Datos $X = \{\mathbf{x}_1, ..., \mathbf{x}_n\}$, número de clusters $K$

**Inicialización:**
$$
\text{Seleccionar } K \text{ centroides iniciales } \{\mu_1^{(0)}, ..., \mu_K^{(0)}\} \tag{3}
$$

**Repetir hasta convergencia:**

**Paso 1: Asignar cada punto al cluster más cercano**
$$
C_k^{(t)} = \{\mathbf{x}_i : \|\mathbf{x}_i - \mu_k^{(t)}\| \leq \|\mathbf{x}_i - \mu_j^{(t)}\| \text{ para todo } j\} \tag{4}
$$

**Paso 2: Actualizar centroides como el promedio de puntos asignados**
$$
\mu_k^{(t+1)} = \frac{1}{|C_k^{(t)}|} \sum_{\mathbf{x}_i \in C_k^{(t)}} \mathbf{x}_i \tag{5}
$$

**Criterio de convergencia:**
- Centroides no cambian: $\mu_k^{(t+1)} = \mu_k^{(t)}$ para todo $k$
- O número máximo de iteraciones alcanzado

---

## 3.3 Propiedades Matemáticas

### Garantía de Convergencia

**Teorema:** K-Means siempre converge a un mínimo local.

**Prueba (sketch):**
1. Cada paso de asignación reduce (o mantiene) $J$
2. Cada paso de actualización de centroides reduce (o mantiene) $J$
3. $J$ está acotado por abajo (≥ 0)
4. Hay un número finito de asignaciones posibles
5. Por lo tanto, debe converger

**Pero:** No garantiza el mínimo global (depende de inicialización).

### Complejidad Computacional

- **Por iteración:** $O(n \times K \times d)$
  - $n$: número de puntos
  - $K$: número de clusters
  - $d$: dimensionalidad

- **Total:** $O(n \times K \times d \times i)$
  - $i$: número de iteraciones (típicamente < 100)

**Conclusión:** Muy eficiente para grandes datasets.

---

## 3.4 K-Means++ (Inicialización Inteligente)

**Problema:** Inicialización aleatoria puede llevar a mal resultado.

**Solución:** Elegir centroides iniciales que estén lejos entre sí.

**Algoritmo:**

1. Elegir primer centroide uniformemente al azar:
$$
\mu_1 \sim \text{Uniform}(X) \tag{6}
$$

2. Para $k = 2, ..., K$:
   - Para cada punto $\mathbf{x}_i$, calcular distancia al centroide más cercano:
$$
D(\mathbf{x}_i) = \min_{j < k} \|\mathbf{x}_i - \mu_j\|^2 \tag{7}
$$
   
   - Elegir $\mu_k$ con probabilidad proporcional a $D(\mathbf{x}_i)^2$:
$$
P(\mathbf{x}_i \text{ es elegido}) = \frac{D(\mathbf{x}_i)^2}{\sum_j D(\mathbf{x}_j)^2} \tag{8}
$$

**Resultado:** Centroides iniciales bien espaciados → mejor convergencia.

---

## 3.5 Elegir K: Método del Codo

**Problema:** ¿Cuántos clusters hay realmente?

**Método del Codo:**
1. Entrenar K-Means para $K = 1, 2, 3, ..., K_{max}$
2. Calcular inercia (suma de distancias al cuadrado)
3. Plotear K vs inercia
4. Buscar el "codo" (punto donde la mejora marginal disminuye)

**Inercia vs K:**
```
Inercia
  |
  |●  
  | ●
  |  ●___●___●___●  ← Codo aquí (K=3)
  |________________ K
   1  2  3  4  5  6
```

### Ejemplo Numérico

**Datos:** 2D, 9 puntos

| Punto | x | y |
|-------|---|---|
| 1 | 1 | 1 |
| 2 | 1 | 2 |
| 3 | 2 | 1 |
| 4 | 5 | 5 |
| 5 | 5 | 6 |
| 6 | 6 | 5 |
| 7 | 9 | 9 |
| 8 | 9 | 10 |
| 9 | 10 | 9 |

**K=3:**
- Cluster 1: {1, 2, 3} → $\mu_1 = (1.33, 1.33)$
- Cluster 2: {4, 5, 6} → $\mu_2 = (5.33, 5.33)$
- Cluster 3: {7, 8, 9} → $\mu_3 = (9.33, 9.33)$

**Inercia:**
$$
J = \sum_{\text{cluster 1}} d^2 + \sum_{\text{cluster 2}} d^2 + \sum_{\text{cluster 3}} d^2 = 2.0 + 2.0 + 2.0 = 6.0
$$

In [ ]:
# Verificar ejemplo numérico
X_example = np.array([
    [1, 1], [1, 2], [2, 1],
    [5, 5], [5, 6], [6, 5],
    [9, 9], [9, 10], [10, 9]
])

kmeans_ex = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_ex.fit(X_example)

print("🔢 Ejemplo Numérico:\n")
print("Centroides:")
for i, centroid in enumerate(kmeans_ex.cluster_centers_):
    print(f"  Cluster {i+1}: {centroid}")

print(f"\nInercia total: {kmeans_ex.inertia_:.2f}")
print("\n💡 Verifica que coincida con nuestro cálculo manual")

---
## 💻 4. Implementación Desde Cero

In [ ]:
class KMeansClustering:
    """
    Implementación desde cero de K-Means clustering.
    
    Parameters:
    -----------
    n_clusters : int, default=3
        Número de clusters a formar
    max_iter : int, default=300
        Número máximo de iteraciones
    tol : float, default=1e-4
        Tolerancia para declarar convergencia
    init : str, default='kmeans++'
        Método de inicialización: 'random' o 'kmeans++'
    random_state : int, default=None
        Semilla para reproducibilidad
    """
    
    def __init__(self, n_clusters=3, max_iter=300, tol=1e-4, 
                 init='kmeans++', random_state=None):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.init = init
        self.random_state = random_state
        
        # Se inicializan durante fit
        self.centroids = None
        self.labels = None
        self.inertia = None
        self.n_iter = 0
    
    def _initialize_centroids(self, X):
        """Inicializa centroides"""
        if self.random_state is not None:
            np.random.seed(self.random_state)
        
        n_samples = X.shape[0]
        
        if self.init == 'random':
            # Selección aleatoria
            indices = np.random.choice(n_samples, self.n_clusters, replace=False)
            centroids = X[indices].copy()
        
        elif self.init == 'kmeans++':
            # K-Means++ initialization
            centroids = []
            
            # Primer centroide aleatorio
            first_idx = np.random.randint(n_samples)
            centroids.append(X[first_idx])
            
            # Siguientes centroides
            for _ in range(1, self.n_clusters):
                # Calcular distancia al centroide más cercano
                distances = np.array([min([np.linalg.norm(x - c)**2 
                                          for c in centroids]) 
                                     for x in X])
                
                # Probabilidad proporcional a distancia^2
                probabilities = distances / distances.sum()
                
                # Seleccionar siguiente centroide
                next_idx = np.random.choice(n_samples, p=probabilities)
                centroids.append(X[next_idx])
            
            centroids = np.array(centroids)
        
        return centroids
    
    def _assign_clusters(self, X, centroids):
        """
        Asigna cada punto al cluster más cercano.
        
        Returns:
        --------
        labels : array, shape (n_samples,)
            Índice del cluster para cada punto
        """
        # Calcular distancias a todos los centroides
        distances = cdist(X, centroids, metric='euclidean')
        
        # Asignar al centroide más cercano
        labels = np.argmin(distances, axis=1)
        
        return labels
    
    def _update_centroids(self, X, labels):
        """
        Actualiza centroides como el promedio de puntos asignados.
        
        Returns:
        --------
        centroids : array, shape (n_clusters, n_features)
        """
        centroids = np.zeros((self.n_clusters, X.shape[1]))
        
        for k in range(self.n_clusters):
            # Puntos asignados al cluster k
            cluster_points = X[labels == k]
            
            if len(cluster_points) > 0:
                # Nuevo centroide = promedio de puntos
                centroids[k] = cluster_points.mean(axis=0)
            else:
                # Si cluster vacío, mantener centroide anterior
                centroids[k] = self.centroids[k]
        
        return centroids
    
    def _calculate_inertia(self, X, labels, centroids):
        """Calcula inercia (suma de distancias al cuadrado)"""
        inertia = 0
        for k in range(self.n_clusters):
            cluster_points = X[labels == k]
            if len(cluster_points) > 0:
                inertia += np.sum((cluster_points - centroids[k])**2)
        return inertia
    
    def fit(self, X):
        """
        Entrena el modelo K-Means.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
        
        Returns:
        --------
        self : KMeansClustering
        """
        # Inicializar centroides
        self.centroids = self._initialize_centroids(X)
        
        print(f"🚀 Ejecutando K-Means con {self.n_clusters} clusters...")
        print(f"   Inicialización: {self.init}\n")
        
        # Iteración
        for iteration in range(self.max_iter):
            # Paso 1: Asignar clusters
            labels = self._assign_clusters(X, self.centroids)
            
            # Paso 2: Actualizar centroides
            new_centroids = self._update_centroids(X, labels)
            
            # Chequear convergencia
            centroid_shift = np.linalg.norm(new_centroids - self.centroids)
            
            if iteration % 10 == 0:
                inertia = self._calculate_inertia(X, labels, new_centroids)
                print(f"   Iteración {iteration:3d}: Inercia = {inertia:10.2f}, "
                      f"Shift = {centroid_shift:.6f}")
            
            if centroid_shift < self.tol:
                print(f"\n✅ Convergencia alcanzada en iteración {iteration}")
                break
            
            self.centroids = new_centroids
            self.n_iter = iteration + 1
        
        # Asignación final
        self.labels = self._assign_clusters(X, self.centroids)
        self.inertia = self._calculate_inertia(X, self.labels, self.centroids)
        
        print(f"\n📊 Inercia final: {self.inertia:.2f}")
        print(f"   Iteraciones: {self.n_iter}")
        
        return self
    
    def predict(self, X):
        """
        Predice cluster para nuevos puntos.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
        
        Returns:
        --------
        labels : array, shape (n_samples,)
        """
        return self._assign_clusters(X, self.centroids)
    
    def fit_predict(self, X):
        """Fit y predict en un solo paso"""
        self.fit(X)
        return self.labels

print("✅ Clase KMeansClustering definida")

In [ ]:
# Probar nuestra implementación
X_test, _ = make_blobs(n_samples=300, centers=4, n_features=2,
                      cluster_std=0.60, random_state=42)

# Entrenar nuestro K-Means
kmeans_custom = KMeansClustering(n_clusters=4, init='kmeans++', random_state=42)
clusters_custom = kmeans_custom.fit_predict(X_test)

In [ ]:
# Visualizar resultado
fig = go.Figure()

colors = ['red', 'blue', 'green', 'orange']
for i in range(4):
    mask = clusters_custom == i
    fig.add_trace(go.Scatter(
        x=X_test[mask, 0],
        y=X_test[mask, 1],
        mode='markers',
        marker=dict(size=8, color=colors[i], opacity=0.6),
        name=f'Cluster {i+1}'
    ))

# Centroides
fig.add_trace(go.Scatter(
    x=kmeans_custom.centroids[:, 0],
    y=kmeans_custom.centroids[:, 1],
    mode='markers',
    marker=dict(size=20, color='black', symbol='x', line=dict(width=2)),
    name='Centroides'
))

fig.update_layout(
    title="Nuestra Implementación de K-Means",
    xaxis_title="Feature 1",
    yaxis_title="Feature 2",
    template="plotly_white",
    font=dict(size=12),
    width=700,
    height=500
)

fig.show()

---
## 🏭 5. Versión con Framework (Scikit-learn)

In [ ]:
# Comparar con Scikit-learn
kmeans_sklearn = KMeans(n_clusters=4, init='k-means++', random_state=42, n_init=10)
clusters_sklearn = kmeans_sklearn.fit_predict(X_test)

print("📊 Comparación: Nuestra Implementación vs Scikit-learn\n")
print("="*60)
print(f"{'Métrica':<30} {'Nuestra':<15} {'Scikit-learn':<15}")
print("="*60)
print(f"{'Inercia':<30} {kmeans_custom.inertia:<15.2f} {kmeans_sklearn.inertia_:<15.2f}")
print(f"{'Iteraciones':<30} {kmeans_custom.n_iter:<15} {kmeans_sklearn.n_iter_:<15}")
print("="*60)

print("\n✅ ¡Resultados similares! Nuestra implementación funciona correctamente")

In [ ]:
# Método del Codo para elegir K
inertias = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_test)
    inertias.append(kmeans.inertia_)

# Plotear
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(K_range),
    y=inertias,
    mode='lines+markers',
    marker=dict(size=10, color='blue'),
    line=dict(color='blue', width=2)
))

fig.update_layout(
    title="Método del Codo: Elegir K Óptimo",
    xaxis_title="Número de Clusters (K)",
    yaxis_title="Inercia",
    template="plotly_white",
    font=dict(size=12),
    annotations=[
        dict(
            x=4, y=inertias[3],
            text="← Codo (K=4)",
            showarrow=True,
            arrowhead=2,
            ax=40, ay=-40
        )
    ]
)

fig.show()

print("\n💡 Busca el 'codo' donde la mejora marginal disminuye")
print("   En este caso, K=4 parece óptimo")

In [ ]:
# Aplicación real: Segmentación de clientes
# Cargar dataset Iris como ejemplo
iris = load_iris()
X_iris = iris.data

# Normalizar (importante para K-Means)
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Aplicar K-Means
kmeans_iris = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters_iris = kmeans_iris.fit_predict(X_iris_scaled)

# Evaluar con métricas
silhouette = silhouette_score(X_iris_scaled, clusters_iris)
davies_bouldin = davies_bouldin_score(X_iris_scaled, clusters_iris)

print("🌸 Clustering del Dataset Iris\n")
print(f"Número de clusters: 3")
print(f"Inercia: {kmeans_iris.inertia_:.2f}")
print(f"\nMétricas de Calidad:")
print(f"  Silhouette Score: {silhouette:.3f} (mejor: 1.0)")
print(f"  Davies-Bouldin Index: {davies_bouldin:.3f} (mejor: 0.0)")

# Comparar con labels verdaderas
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(iris.target, clusters_iris)
print(f"  Adjusted Rand Index: {ari:.3f} (vs labels verdaderas)")

print("\n💡 K-Means descubrió clusters muy similares a las especies reales")

### Comparación: K-Means vs Otros Métodos de Clustering

| Método | Ventajas | Desventajas | Cuándo usar |
|--------|----------|-------------|-------------|
| **K-Means** | Rápido, escalable, simple | K fijo, clusters esféricos, sensible a outliers | Datos bien separados, K conocido |
| **DBSCAN** | Descubre K, maneja ruido, formas arbitrarias | Sensible a densidad variable | Datos con ruido, formas irregulares |
| **Hierarchical** | No requiere K, dendrograma | Lento (O(n³)), no escalable | Exploración, visualización |
| **GMM** | Clusters probabilísticos, soft assignment | Más complejo, más lento | Clusters superpuestos |

### Limitaciones de K-Means

1. **Asume clusters esféricos y del mismo tamaño**
2. **Sensible a inicialización** (solución: K-Means++, múltiples runs)
3. **Sensible a outliers** (un punto lejano afecta el centroide)
4. **Requiere especificar K de antemano**
5. **Solo distancia Euclidiana** (en implementación estándar)

---
## 🎯 6. Ejercicios

### 🟢 Ejercicio 1: Compresión de Imágenes

Usa K-Means para reducir la paleta de colores de una imagen.

In [ ]:
def ejercicio_1():
    """
    Objetivo: Aplicar K-Means a compresión de imágenes
    
    Instrucciones:
    1. Crea una imagen sintética o carga una pequeña (ej: digits dataset)
    2. Cada pixel es un punto en espacio de color (R, G, B)
    3. Aplica K-Means con K=8, 16, 32 colores
    4. Reemplaza cada pixel por el centroide más cercano
    5. Visualiza: imagen original vs comprimidas
    6. Calcula: ratio de compresión (bits originales / bits comprimidos)
    
    Returns:
    --------
    compression_ratios : dict
        {n_colors: ratio}
    """
    # TODO: Tu código aquí
    # Pista: usa load_digits() de sklearn y trata cada pixel como feature
    
    pass

# Descomentar para probar
# ratios = ejercicio_1()
# print("\n💡 Menos colores = mayor compresión pero menor calidad")

### 🟡 Ejercicio 2: Comparación Random Init vs K-Means++

Demuestra empíricamente que K-Means++ es mejor.

In [ ]:
def ejercicio_2():
    """
    Objetivo: Comparar métodos de inicialización
    
    Instrucciones:
    1. Genera datos con make_blobs(n_samples=1000, centers=5)
    2. Para cada método ('random', 'k-means++'):
       a) Ejecuta K-Means 20 veces (diferentes random_state)
       b) Registra: inercia final, número de iteraciones
    3. Compara estadísticas:
       - Inercia promedio
       - Iteraciones promedio
       - Desviación estándar de inercia
    4. Visualiza distribución de inertias (boxplot)
    
    Returns:
    --------
    results : dict
        Estadísticas de cada método
    """
    # TODO: Tu código aquí
    
    pass

# Descomentar para probar
# results = ejercicio_2()
# print("\n💡 K-Means++ debería tener menor varianza y mejor inercia promedio")

### 🔴 Ejercicio 3: K-Means con Restricciones

Implementa una variante de K-Means donde los clusters deben tener tamaño mínimo/máximo.

In [ ]:
def ejercicio_3():
    """
    Objetivo: Implementar Constrained K-Means
    
    Instrucciones:
    1. Modifica el algoritmo K-Means para que:
       - Cada cluster tenga al menos min_size puntos
       - Cada cluster tenga a lo más max_size puntos
    2. Algoritmo:
       a) Asignar normalmente
       b) Si cluster viola restricciones:
          - Si muy pequeño: asignar puntos más cercanos de otros clusters
          - Si muy grande: reasignar puntos más lejanos a otros clusters
    3. Prueba en dataset sintético desbalanceado
    4. Compara con K-Means estándar:
       - Distribución de tamaños de clusters
       - Inercia
    
    Returns:
    --------
    comparison : dict
        Comparación de métricas
    """
    # TODO: Tu código aquí
    # Pista: Este es un problema NP-hard, usa heurística iterativa
    
    pass

# Descomentar para probar
# comparison = ejercicio_3()
# print("\n🎯 K-Means con restricciones es útil para balancear cargas")

---
## 📚 7. Resumen y Recursos

### 🎯 Puntos Clave

1. **K-Means es clustering basado en centroides**
   - Algoritmo iterativo: asignar → actualizar → repetir
   - Minimiza inercia (suma de distancias al cuadrado)
   - Converge a un mínimo local (no necesariamente global)

2. **Funcionamiento simple pero efectivo**
   - Solo requiere calcular distancias y promedios
   - Complejidad O(n × K × d × i) - muy escalable
   - Típicamente converge en < 100 iteraciones

3. **K-Means++ mejora inicialización**
   - Espaciar centroides iniciales
   - Garantía teórica: O(log K)-aproximación
   - Significativamente más estable que random init

4. **Método del codo para elegir K**
   - Plotear K vs inercia
   - Buscar punto donde mejora marginal disminuye
   - Complementar con métricas: Silhouette, Davies-Bouldin

5. **Aplicaciones prácticas:**
   - Segmentación de clientes
   - Compresión de imágenes (reducción de colores)
   - Preprocessing para otros algoritmos
   - Análisis exploratorio de datos

6. **Ventajas:**
   - ✅ Simple de entender e implementar
   - ✅ Muy rápido y escalable
   - ✅ Funciona bien con clusters esféricos
   - ✅ Fácil de paralelizar

7. **Limitaciones:**
   - ❌ Requiere especificar K de antemano
   - ❌ Asume clusters esféricos de tamaño similar
   - ❌ Sensible a outliers
   - ❌ Sensible a inicialización (mitigado con K-Means++)
   - ❌ Solo distancia Euclidiana (en versión estándar)

8. **Buenas prácticas:**
   - **Normalizar datos** (StandardScaler)
   - Usar **K-Means++** para inicialización
   - Ejecutar **múltiples veces** con diferentes seeds
   - Probar **varios valores de K**
   - Validar con **métricas de clustering**

---

### 🔗 Recursos Adicionales

#### 📄 Papers Fundamentales

- **"Least squares quantization in PCM"** - Stuart Lloyd (1982)
  - Paper original del algoritmo (propuesto en 1957)
  - IEEE Transactions on Information Theory

- **"k-means++: The Advantages of Careful Seeding"** - Arthur & Vassilvitskii (2007)
  - Introduce K-Means++
  - Garantías teóricas de aproximación
  - http://ilpubs.stanford.edu:8090/778/1/2006-13.pdf

- **"A Survey of Clustering Data Mining Techniques"** - Berkhin (2006)
  - Overview completo de métodos de clustering

#### 📖 Libros Recomendados

- **"Pattern Recognition and Machine Learning"** - Bishop
  - Capítulo 9: Mixture Models and EM
  - K-Means como caso especial de GMM

- **"Data Mining: Concepts and Techniques"** - Han, Kamber, Pei
  - Capítulos 10-11: Clustering
  - Muy aplicado y práctico

- **"The Elements of Statistical Learning"** - Hastie et al.
  - Capítulo 14.3: K-Means Clustering

#### 🎥 Videos Recomendados

- **StatQuest: K-Means Clustering** - Josh Starmer
  - Visualización excelente del algoritmo
  - https://www.youtube.com/watch?v=4b5d3muPQmA

- **Stanford CS229: Clustering** - Andrew Ng
  - Tratamiento matemático riguroso

#### 💻 Documentación y Tutoriales

- [Scikit-learn: K-Means](https://scikit-learn.org/stable/modules/clustering.html#k-means)
- [Visualizing K-Means Clustering](https://www.naftaliharris.com/blog/visualizing-k-means-clustering/)
- [Choosing the Number of Clusters](https://towardsdatascience.com/10-tips-for-choosing-the-optimal-number-of-clusters-277e93d72d92)

#### 🧪 Recursos Interactivos

- [K-Means Visualizer](https://www.naftaliharris.com/blog/visualizing-k-means-clustering/) by Naftali Harris
- [Clustering Playground](https://user.ceng.metu.edu.tr/~akifakkus/courses/ceng574/)

---

### 🤔 Preguntas para Reflexionar

1. **¿Por qué K-Means minimiza inercia pero no error de clasificación?**
   - Pista: No hay labels en clustering

2. **¿K-Means puede usarse para compresión de datos?**
   - Piensa en cuantización de vectores

3. **¿Qué pasa si hay outliers extremos?**
   - Considera K-Medians como alternativa

4. **¿Cómo adaptarías K-Means para datos categóricos?**
   - Pista: K-Modes usa "modo" en lugar de "media"

5. **¿K-Means puede encontrar clusters no convexos?**
   - Considera kernel K-Means

---

## ➡️ Próximo Paso

En el siguiente notebook, **10. PCA (Principal Component Analysis)**, aprenderemos sobre:

- **Reducción de dimensionalidad**: De alta a baja dimensión
- **Eigenvalues y eigenvectors**: Álgebra lineal aplicada
- **Varianza explicada**: Cuánta información retener
- **Proyección**: Transformar datos a nuevo espacio
- **Visualización**: Graficar datos de alta dimensión

**Conexión:** K-Means agrupa puntos similares, PCA encuentra direcciones de máxima varianza.

---

<div align="center">

**🎯 De agrupar puntos a reducir dimensiones 🎯**

**Continúa con: [10. PCA](10-pca.ipynb)**

[← 08. SVM](08-svm.ipynb) | [10. PCA →](10-pca.ipynb)

</div>